<a href="https://colab.research.google.com/github/thequantumbruh/LLMs/blob/main/LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torch.utils.data import dataloader, dataset
import tiktoken
import torch.nn as nn

In [2]:
from tiktoken.registry import get_encoding
tokenizer = tiktoken.get_encoding("gpt2")

In [3]:
class LLM_dataset:
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_id = []
    self.target_id = []
    self.tokens = tokenizer.encode(txt)

    for i in range (0, len(self.tokens) - max_length, stride):
      input_chunk = self.tokens[i:i + max_length]
      target_chunk = self.tokens[i+1: max_length+1]
      self.input_id.append(torch.tensor(input_chunk))#Remember to always convert your ist to tensors
      self.target_id.append(torch.tensor(target_chunk))

  def __len__ (self):
    return len(self.input_id)

  def __getitem__(self, idx):
    return self.input_id[idx], self.target_id[idx]


In [4]:
def load_data(text:str ,max_length:int, stride:int):
  tokenizer = tiktoken.get_encoding("gpt2")
  llm_ds = LLM_dataset(text, tokenizer, 4, 1)

  dataloader = torch.utils.data.DataLoader(dataset= llm_ds, num_workers = 8)
  return dataloader


In [5]:
import requests
url = "https://www.gutenberg.org/cache/epub/64317/pg64317.txt"
response = requests.get(url)
text = response.text

with open("the gutenberg article.txt", "w", encoding = "utf-8") as f:
  f.write(text)
txt = "the gutenberg article.txt"
data_iter = iter(load_data(txt, 4, 1))
inputs, targets= next(data_iter)
print (f"inputs:   ${inputs}")

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


inputs:   $tensor([[ 1169,   308, 19028,  2708]])


In [6]:
print(inputs.size())
size = inputs.size()
embedding = torch.nn.Embedding(size[-1], 21)

torch.Size([1, 4])


In [7]:
#Attention mechanism

class attn_mech (nn.Module):
  def __init__(self, d_in: int, d_out: int, qkv_bias = False):
    super().__init__()

    self.v = nn.Linear(d_out, d_out, bias = qkv_bias)
    self.q = nn.Linear(d_out, d_out, bias = qkv_bias)
    self.k = nn.Linear(d_out, d_out, bias = qkv_bias)

  def foward(self, x):
      batch_size, token_num, d_in = x.shape()
      self.query = self.q(x)
      self.key = self.k(x)
      self.value = self.v(x)

      attn_weights = torch.nn.Softmax((self.query @ self.key.transpose(1,2))/x.shape[-1]**0.5)
      context_vector = attn_weights @ self.value

      return context_vector


In [10]:
from torch._C import double

#implementing attention mechanism while applying mask "Causal attention"

class causal_attn (nn.Module):
  def __init__ (self,d_in: int, d_out: int, dropout: double, context_len, qkv_bias = False):
    super().__init__()
    self.W_q = nn.Linear(d_in, d_out, bias= qkv_bias)
    self.W_k = nn.Linear(d_in, d_out, bias= qkv_bias)
    self.W_v = nn.Linear(d_in, d_out, bias= qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.nn.Module.register_buffer("mask",
                                   torch.triu(torch.ones(context_len, context_len), diagonal = 1))

  def foward(self, x):
    batch, tokn_num, emb_dim = x.shape()
    mask = torch.triu(torch.ones(tokn_num, emb_dim), 1)
    self.query = self.W_q(x)
    self.key = self.W_k(x)
    self.value = self.W_v(x)
    attn_scores = nn.Softmax((self.query @ self.key.transpose(1,2))/x.shape[-1]**0.5)
    x = self.dropout(x)
    attn_scores = attn_scores.masked_fill(mask.bool(), -torch.inf)
    attn_weight = attn_scores/(emb_dim**0.5)
    context_vector = attn_weight * self.value

    return context_vector


In [14]:
torch.mas

Help on function register_buffer in module torch.nn.modules.module:

register_buffer(
    self,
    name: str,
    tensor: torch.Tensor | None,
    persistent: bool = True
) -> None
    Add a buffer to the module.

    This is typically used to register a buffer that should not be
    considered a model parameter. For example, BatchNorm's ``running_mean``
    is not a parameter, but is part of the module's state. Buffers, by
    default, are persistent and will be saved alongside parameters. This
    behavior can be changed by setting :attr:`persistent` to ``False``. The
    only difference between a persistent buffer and a non-persistent buffer
    is that the latter will not be a part of this module's
    :attr:`state_dict`.

    Buffers can be accessed as attributes using given names.

    Args:
        name (str): name of the buffer. The buffer can be accessed
            from this module using the given name
        tensor (Tensor or None): buffer to be registered. If ``None``, th